## Phase 2: Concept Drift Detectors
### Step 1 — Data Initialization & Multivariate Structure Validation

Loads preprocessed IHSG data, validates datetime index, builds drift scoreboard
DataFrames (one per algorithm variant) with NaN for unevaluable initial rows.

#### 1. Configuration & Imports

Defines `Config` dataclass holding all tunable parameters:
- Data paths, feature names, batch/stream algorithm lists
- Window sizes (60, 120), stream warm-up period (60 rows)
- Consensus threshold (30%) for voting in later steps

In [1]:
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd


@dataclass
class Config:
    data_path: str = "../dataset/processed/jkse_preprocessed.csv"
    output_dir: str = "../dataset/processed"
    features: list = field(default_factory=lambda: [
        "Log_Return", "Vol_20d", "Vol_60d", "EMA_5",
        "BB_Middle", "BB_Upper", "BB_Lower",
        "Momentum_5d", "Momentum_20d",
    ])
    batch_windows: list = field(default_factory=lambda: [60, 120])
    batch_algorithms: list = field(default_factory=lambda: [
        "mysd", "ks", "psi", "wasserstein",
    ])
    stream_algorithms: list = field(default_factory=lambda: [
        "adwin", "kswin", "ph",
    ])
    stream_warmup: int = 60
    consensus_threshold: float = 0.3

#### 2. Data Loading & Validation

Loads preprocessed JKSE CSV, parses DatetimeIndex, sorts chronologically, and runs sanity assertions:
- Index is unique and datetime-typed
- No remaining NaN values after preprocessing
- Prints date range and shape

In [2]:
cfg = Config()

print("Config:")
for k, v in cfg.__dict__.items():
    val = str(v)
    if len(val) > 80:
        val = val[:77] + "..."
    print(f"  {k:20s} = {val}")

df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True)
df = df.sort_index()

assert isinstance(df.index, pd.DatetimeIndex), "Index must be DatetimeIndex"
assert df.index.is_unique, "Index contains duplicate dates"
assert df.isnull().sum().sum() == 0, "Data contains NaN values"

print(f"\nLoaded {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Index type: {df.index.dtype}")
print(f"Date range: {df.index.min().date()}  to  {df.index.max().date()}")

Config:
  data_path            = ../dataset/processed/jkse_preprocessed.csv
  output_dir           = ../dataset/processed
  features             = ['Log_Return', 'Vol_20d', 'Vol_60d', 'EMA_5', 'BB_Middle', 'BB_Upper', 'BB_Lo...
  batch_windows        = [60, 120]
  batch_algorithms     = ['mysd', 'ks', 'psi', 'wasserstein']
  stream_algorithms    = ['adwin', 'kswin', 'ph']
  stream_warmup        = 60
  consensus_threshold  = 0.3

Loaded 3930 rows x 14 columns
Index type: datetime64[us]
Date range: 2010-03-31  to  2026-06-19


#### 3. Scoreboard Creation

Creates one scoreboard per algorithm-variant (11 total):
- **Batch** (`mysd`, `ks`, `psi`, `wasserstein` × 60, 120): `NaN` for first `2*W` rows (insufficient data for two adjacent windows)
- **Stream** (`adwin`, `kswin`, `ph`): `NaN` for first 60 warm-up rows
- All remaining rows initialized to `0.0` (no drift detected yet)

In [3]:
def create_scoreboards(df: pd.DataFrame, cfg: Config) -> dict[str, pd.DataFrame]:
    scoreboards: dict[str, pd.DataFrame] = {}

    for alg in cfg.batch_algorithms:
        for w in cfg.batch_windows:
            key = f"{alg}_{w}"
            sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
            sb.iloc[2 * w:] = 0.0
            scoreboards[key] = sb

    for alg in cfg.stream_algorithms:
        sb = pd.DataFrame(np.nan, index=df.index, columns=cfg.features)
        sb.iloc[cfg.stream_warmup:] = 0.0
        scoreboards[alg] = sb

    return scoreboards


def scoreboard_summary(scoreboards: dict[str, pd.DataFrame], cfg: Config) -> str:
    lines = [f"Scoreboards created: {len(scoreboards)}"]
    for key, sb in scoreboards.items():
        nan_rows = int(sb.isnull().all(axis=1).sum())
        lines.append(f"  {key:20s}  shape={str(sb.shape):12s}  NaN rows={nan_rows:>4d}")
    return "\n".join(lines)


scoreboards = create_scoreboards(df, cfg)
print("=" * 55)
print("STEP 1 - Result")
print("=" * 55)
print(f"df.shape:              {df.shape}")
print(f"Feature columns:       {cfg.features}")
print(f"Batch windows (days):  {cfg.batch_windows}")
print(f"Stream algorithms:     {cfg.stream_algorithms}")
print(f"Stream warm-up rows:   {cfg.stream_warmup}")
print()
print(scoreboard_summary(scoreboards, cfg))

STEP 1 - Result
df.shape:              (3930, 14)
Feature columns:       ['Log_Return', 'Vol_20d', 'Vol_60d', 'EMA_5', 'BB_Middle', 'BB_Upper', 'BB_Lower', 'Momentum_5d', 'Momentum_20d']
Batch windows (days):  [60, 120]
Stream algorithms:     ['adwin', 'kswin', 'ph']
Stream warm-up rows:   60

Scoreboards created: 11
  mysd_60               shape=(3930, 9)     NaN rows= 120
  mysd_120              shape=(3930, 9)     NaN rows= 240
  ks_60                 shape=(3930, 9)     NaN rows= 120
  ks_120                shape=(3930, 9)     NaN rows= 240
  psi_60                shape=(3930, 9)     NaN rows= 120
  psi_120               shape=(3930, 9)     NaN rows= 240
  wasserstein_60        shape=(3930, 9)     NaN rows= 120
  wasserstein_120       shape=(3930, 9)     NaN rows= 240
  adwin                 shape=(3930, 9)     NaN rows=  60
  kswin                 shape=(3930, 9)     NaN rows=  60
  ph                    shape=(3930, 9)     NaN rows=  60


#### 4. Feature Diagnostics

Exploratory analysis of the 9 feature columns:
- Summary statistics via `describe().T`
- Correlation matrix with heatmap coloring (`RdBu_r`)
- Memory footprint and missing-value check

In [4]:
print("Feature summary statistics")
print("-" * 55)
display(df[cfg.features].describe().T)

print("\nFeature correlation matrix")
print("-" * 55)
corr = df[cfg.features].corr()
display(corr.style.background_gradient(cmap="RdBu_r", vmin=-1, vmax=1))

missing = int(df[cfg.features].isnull().sum().sum())
print(f"\nMissing values in feature set: {missing}")
print(f"DataFrame memory usage: {df[cfg.features].memory_usage(deep=True).sum() / 1024:.1f} KB")

Feature summary statistics
-------------------------------------------------------


,count,mean,std,min,25%,50%,75%,max
Log_Return,3930.0,0.000201,0.010921,-0.092997,-0.004761,0.000787,0.005850,0.097042
Vol_20d,3930.0,0.009599,0.005051,0.003133,0.006534,0.008272,0.010679,0.042301
Vol_60d,3930.0,0.009857,0.004299,0.003955,0.006932,0.008532,0.011226,0.027811
EMA_5,3930.0,5615.936638,1293.543632,2625.637390,4589.898093,5773.503558,6661.949826,9071.284552
BB_Middle,3930.0,5615.912165,1294.082105,2627.493555,4586.577734,5763.981445,6662.240186,9077.378711
BB_Upper,3930.0,5709.857326,1307.191864,2777.316847,4704.031722,5850.919315,6742.263439,9453.328998
BB_Lower,3930.0,5521.967004,1285.032770,2450.216971,4487.974647,5647.314534,6563.320200,8963.434251
Momentum_5d,3930.0,0.001006,0.024432,-0.176059,-0.009310,0.002655,0.013967,0.157750
Momentum_20d,3930.0,0.004101,0.047435,-0.385058,-0.015626,0.009183,0.031409,0.154075



Feature correlation matrix
-------------------------------------------------------


,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Log_Return,1.000000,0.005858,0.007892,-0.026928,-0.032930,-0.039164,-0.026484,0.433064,0.211536
Vol_20d,0.005858,1.000000,0.773502,-0.262658,-0.263596,-0.231420,-0.295494,-0.044830,-0.390727
Vol_60d,0.007892,0.773502,1.000000,-0.334359,-0.334733,-0.310800,-0.358020,-0.010351,-0.153116
EMA_5,-0.026928,-0.262658,-0.334359,1.000000,0.999912,0.998362,0.998330,-0.027946,-0.019105
BB_Middle,-0.032930,-0.263596,-0.334733,0.999912,1.000000,0.998460,0.998407,-0.033245,-0.018927
BB_Upper,-0.039164,-0.231420,-0.310800,0.998362,0.998460,1.000000,0.993739,-0.050087,-0.043661
BB_Lower,-0.026484,-0.295494,-0.358020,0.998330,0.998407,0.993739,1.000000,-0.016008,0.006293
Momentum_5d,0.433064,-0.044830,-0.010351,-0.027946,-0.033245,-0.050087,-0.016008,1.000000,0.481258
Momentum_20d,0.211536,-0.390727,-0.153116,-0.019105,-0.018927,-0.043661,0.006293,0.481258,1.000000



Missing values in feature set: 0
DataFrame memory usage: 307.0 KB


#### 5. Persist Scoreboards

Saves all 11 scoreboard DataFrames as CSV to `../dataset/processed/`.
Files are named `scoreboard_{key}.csv` (e.g. `scoreboard_mysd_60.csv`).
These will be loaded by later steps to write actual drift scores into.

In [5]:
import os

os.makedirs(cfg.output_dir, exist_ok=True)

for key, sb in scoreboards.items():
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    sb.to_csv(path)

print(f"Saved {len(scoreboards)} scoreboards to {cfg.output_dir}/")
for key in scoreboards:
    p = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    size_kb = p.stat().st_size / 1024
    print(f"  {key:20s}  {size_kb:>8.1f} KB")

Saved 11 scoreboards to ../dataset/processed/
  mysd_60                  177.3 KB
  mysd_120                 174.1 KB
  ks_60                    177.3 KB
  ks_120                   174.1 KB
  psi_60                   177.3 KB
  psi_120                  174.1 KB
  wasserstein_60           177.3 KB
  wasserstein_120          174.1 KB
  adwin                    178.9 KB
  kswin                    178.9 KB
  ph                       178.9 KB


#### 6. Scoreboard Previews

Displays `head()` of each scoreboard to visually confirm:
- NaN warm-up rows for batch (120 or 240) and stream (60) algorithms
- Correct column structure (9 features)
- Zeroes after the warm-up period

In [6]:
print("=" * 55)
print("SCOREBOARD PREVIEWS")
print("=" * 55)
for key, sb in scoreboards.items():
    nan_rows = int(sb.isnull().all(axis=1).sum())
    print(f"\n{key}  (NaN rows: {nan_rows})")
    display(sb.head())

SCOREBOARD PREVIEWS

mysd_60  (NaN rows: 120)


,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Date,,,,,,,,,
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



mysd_120  (NaN rows: 240)


,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Date,,,,,,,,,
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



ks_60  (NaN rows: 120)


...

,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Date,,,,,,,,,
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



kswin  (NaN rows: 60)


,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Date,,,,,,,,,
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



ph  (NaN rows: 60)


,Log_Return,Vol_20d,Vol_60d,EMA_5,BB_Middle,BB_Upper,BB_Lower,Momentum_5d,Momentum_20d
Date,,,,,,,,,
2010-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-04-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Step 2 — Priority 1: Basic Statistics (mySD & KS)

Evaluates drift per feature via adjacent-window comparison using:
- **mySD** (custom): $|\\mu_{\\text{curr}} - \\mu_{\\text{ref}}| > k \\cdot \\sigma_{\\text{ref}}$ with $k=2.5$, $\\sigma$ using `ddof=1`
- **Kolmogorov-Smirnov**: `ks_2samp(ref, curr)` with $\\alpha = 0.05$

Slicing: `ref_window = df[feat].iloc[t-2*W : t-W]`, `curr_window = df[feat].iloc[t-W : t]`

Evaluation starts at $t = 2W$ (first row where both windows are full).

In [7]:
from scipy.stats import ks_2samp
import time

MYSD_K = 2.5
KS_ALPHA = 0.05

batch_start = time.perf_counter()

print("=" * 55)
print("STEP 2 — Drift Detection Loop")
print("=" * 55)

for w in cfg.batch_windows:
    print(f"\n  Processing W={w}...")
    for feat in cfg.features:
        # mySD — vectorized via rolling stats
        rolling_mean = df[feat].rolling(w).mean()
        rolling_std = df[feat].rolling(w).std(ddof=1)
        sb_mysd = scoreboards[f"mysd_{w}"]
        for t in range(2 * w, len(df)):
            mu_ref = rolling_mean.iloc[t - w]
            sigma_ref = rolling_std.iloc[t - w]
            if sigma_ref == 0.0:
                continue
            if abs(rolling_mean.iloc[t] - mu_ref) > MYSD_K * sigma_ref:
                sb_mysd.loc[df.index[t], feat] = 1.0

        # KS — direct loop (scipy, no vectorization)
        sb_ks = scoreboards[f"ks_{w}"]
        for t in range(2 * w, len(df)):
            ref = df[feat].iloc[t - 2 * w: t - w]
            cur = df[feat].iloc[t - w: t]
            pvalue = ks_2samp(ref, cur).pvalue
            if not np.isnan(pvalue) and pvalue < KS_ALPHA:
                sb_ks.loc[df.index[t], feat] = 1.0
        print(f"    {feat} done")

batch_elapsed = time.perf_counter() - batch_start
print(f"\nBatch group runtime: {batch_elapsed:.2f}s")

print("\nDetection complete. Persisting updated scoreboards...")

for key in ("mysd_60", "mysd_120", "ks_60", "ks_120"):
    path = Path(cfg.output_dir) / f"scoreboard_{key}.csv"
    scoreboards[key].to_csv(path)
    size_kb = path.stat().st_size / 1024
    print(f"  Saved  {key:20s}  {size_kb:>8.1f} KB")


STEP 2 — Drift Detection Loop

  Processing W=60...
    Log_Return done
    Vol_20d done
    Vol_60d done
    EMA_5 done
    BB_Middle done
    BB_Upper done
    BB_Lower done
    Momentum_5d done
    Momentum_20d done

  Processing W=120...
    Log_Return done
    Vol_20d done
    Vol_60d done
    EMA_5 done
    BB_Middle done
    BB_Upper done
    BB_Lower done
    Momentum_5d done
    Momentum_20d done

Batch group runtime: 148.48s

Detection complete. Persisting updated scoreboards...
  Saved  mysd_60                  177.3 KB
  Saved  mysd_120                 174.1 KB
  Saved  ks_60                    177.3 KB
  Saved  ks_120                   174.1 KB


### Step 2 Results: Basic Statistics Group

Aggregated drift-detected day counts per feature for mySD and KS-Test.

In [8]:
def _drift_report(scoreboards, cfg):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 2 — DRIFT SUMMARY REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append("### Step 2 Results: Basic Statistics Group")
    lines.append("")

    for w, period in [(60, "Quarterly Window (W = 60)"), (120, "Semiannual Window (W = 120)")]:
        lines.append(f"#### {period}")
        lines.append("")
        for alg_key, display_name in [("mysd", "mySD"), ("ks", "KS-Test")]:
            sb = scoreboards[f"{alg_key}_{w}"]
            counts = sb.sum().astype(int)
            header = f"* **{display_name} (W={w}):**"
            items = []
            for col in cfg.features:
                items.append(f"  - {col}: {counts[col]} days")
            lines.append(header)
            lines.extend(items)
            lines.append("")

    return "\n".join(lines)


print(_drift_report(scoreboards, cfg))


STEP 2 — DRIFT SUMMARY REPORT

### Step 2 Results: Basic Statistics Group

#### Quarterly Window (W = 60)

* **mySD (W=60):**
  - Log_Return: 0 days
  - Vol_20d: 685 days
  - Vol_60d: 1551 days
  - EMA_5: 1418 days
  - BB_Middle: 1363 days
  - BB_Upper: 1304 days
  - BB_Lower: 1212 days
  - Momentum_5d: 4 days
  - Momentum_20d: 219 days

* **KS-Test (W=60):**
  - Log_Return: 354 days
  - Vol_20d: 3606 days
  - Vol_60d: 3715 days
  - EMA_5: 3654 days
  - BB_Middle: 3634 days
  - BB_Upper: 3625 days
  - BB_Lower: 3640 days
  - Momentum_5d: 1781 days
  - Momentum_20d: 3182 days

#### Semiannual Window (W = 120)

* **mySD (W=120):**
  - Log_Return: 0 days
  - Vol_20d: 536 days
  - Vol_60d: 974 days
  - EMA_5: 1183 days
  - BB_Middle: 1159 days
  - BB_Upper: 1165 days
  - BB_Lower: 1028 days
  - Momentum_5d: 0 days
  - Momentum_20d: 21 days

* **KS-Test (W=120):**
  - Log_Return: 576 days
  - Vol_20d: 3544 days
  - Vol_60d: 3678 days
  - EMA_5: 3635 days
  - BB_Middle: 3638 days
  - BB_Uppe

### Step 3 — Priority 2: Pure Streaming Group (river)

Shifts from batch rolling-window (Step 2) to pure online stream processing.
Uses `river` library detectors (ADWIN, KSWIN, PageHinkley) that process data
incrementally, one row at a time, in chronological order.

**Key rules:**
- One detector instance per feature, instantiated **outside** the row loop
- 60-row warm-up: first 60 rows remain `NaN` in scoreboard
- Recording starts at row 60 (index 60, 0-based)
- NaN guard skips `.update()` for any missing or invalid values


In [ ]:
from river.drift import ADWIN, KSWIN, PageHinkley
from river.preprocessing import StandardScaler
import time

# Reload stream scoreboards from CSV to guard against kernel state loss
for alg in cfg.stream_algorithms:
    path = Path(cfg.output_dir) / f"scoreboard_{alg}.csv"
    scoreboards[alg] = pd.read_csv(path, index_col=0, parse_dates=True)

print("=" * 55)
print("STEP 3 — Pure Streaming Drift Detection")
print("=" * 55)

# Instantiate one scaler per feature (shared across all detectors)
# Uses transform-then-learn order to avoid look-ahead bias.
# StandardScaler expects dict input, even for single features.
scalers = {feat: StandardScaler() for feat in cfg.features}
print(f"  Initialized {len(scalers)} scalers\n")

# Instantiate one detector per feature — OUTSIDE the row loop
detectors = {}
for alg, constructor in [
    ("adwin", lambda: ADWIN(delta=0.002)),
    ("kswin", lambda: KSWIN(alpha=0.005, window_size=100, stat_size=30)),
    ("ph", lambda: PageHinkley()),
]:
    detectors[alg] = {feat: constructor() for feat in cfg.features}
    print(f"  Initialized {alg}: 9 detectors")

stream_start = time.perf_counter()

for t in range(len(df)):
    for alg in cfg.stream_algorithms:
        for feat in cfg.features:
            val = df[feat].iloc[t]
            if pd.isna(val):
                continue
            # Standardize to Z-score using pre-update running stats
            val_scaled = scalers[feat].transform_one({feat: val})[feat]
            scalers[feat].learn_one({feat: val})
            # Feed standardized value to detector
            detectors[alg][feat].update(val_scaled)
            if t >= cfg.stream_warmup and detectors[alg][feat].drift_detected:
                scoreboards[alg].loc[df.index[t], feat] = 1.0

stream_elapsed = time.perf_counter() - stream_start
print(f"\nStream group runtime: {stream_elapsed:.2f}s")

# Persist updated stream scoreboards
print("\nPersisting updated scoreboards...")
for alg in cfg.stream_algorithms:
    path = Path(cfg.output_dir) / f"scoreboard_{alg}.csv"
    scoreboards[alg].to_csv(path)
    size_kb = path.stat().st_size / 1024
    print(f"  Saved  {alg:20s}  {size_kb:>8.1f} KB")

STEP 3 — Pure Streaming Drift Detection
  Initialized 9 scalers

  Initialized adwin: 9 detectors
  Initialized kswin: 9 detectors
  Initialized ph: 9 detectors

Stream group runtime: 10.85s

Persisting updated scoreboards...
  Saved  adwin                    178.9 KB
  Saved  kswin                    178.9 KB
  Saved  ph                       178.9 KB


### Step 3 Results: Pure Streaming Group (river)

Aggregated drift-detected day counts per feature for ADWIN, KSWIN, and PageHinkley.
Runtime comparison against batch rolling-window methods (Step 2) is included.

In [ ]:
def _stream_drift_report(scoreboards, cfg, stream_elapsed, batch_elapsed):
    lines = []
    lines.append("=" * 55)
    lines.append("STEP 3 — DRIFT SUMMARY REPORT")
    lines.append("=" * 55)
    lines.append("")
    lines.append("### Step 3 Results: Pure Streaming Group (river)")
    lines.append("")
    lines.append("#### A. Drift Signal Summary (Total Days)")
    lines.append("")

    alg_map = {
        "adwin": "ADWIN",
        "kswin": "KSWIN",
        "ph": "Page Hinkley",
    }
    for alg, display_name in alg_map.items():
        sb = scoreboards[alg]
        counts = sb.sum().astype(int)
        header = f"* **{display_name}:**"
        items = []
        for col in cfg.features:
            items.append(f"  - {col}: {counts[col]} days")
        lines.append(header)
        lines.extend(items)
        lines.append("")

    lines.append("#### B. Runtime Comparison & Observations")
    lines.append("")
    lines.append(
        f"* Batch group (rolling windows, mySD + KS-Test):  {batch_elapsed:.2f}s"
    )
    lines.append(
        f"* Stream group (row-by-row, river ADWIN+KSWIN+PH): {stream_elapsed:.2f}s"
    )
    ratio = batch_elapsed / stream_elapsed if stream_elapsed > 0 else float("inf")
    lines.append(f"* River stream group is {ratio:.1f}x faster")
    lines.append("")
    lines.append("> Despite being row-by-row, river detectors are implemented in")
    lines.append("> optimized C/Cython under the hood. The batch group is slower")
    lines.append("> because it uses Python-level loops for KS-2sample tests and")
    lines.append("> manual mySD comparisons across each window step.")

    return "\n".join(lines)


print(_stream_drift_report(scoreboards, cfg, stream_elapsed, batch_elapsed))

STEP 3 — DRIFT SUMMARY REPORT

### Step 3 Results: Pure Streaming Group (river)

#### A. Drift Signal Summary (Total Days)

* **ADWIN:**
  - Log_Return: 1 days
  - Vol_20d: 27 days
  - Vol_60d: 30 days
  - EMA_5: 25 days
  - BB_Middle: 25 days
  - BB_Upper: 26 days
  - BB_Lower: 25 days
  - Momentum_5d: 5 days
  - Momentum_20d: 33 days

* **KSWIN:**
  - Log_Return: 7 days
  - Vol_20d: 38 days
  - Vol_60d: 38 days
  - EMA_5: 39 days
  - BB_Middle: 38 days
  - BB_Upper: 38 days
  - BB_Lower: 38 days
  - Momentum_5d: 28 days
  - Momentum_20d: 37 days

* **Page Hinkley:**
  - Log_Return: 3 days
  - Vol_20d: 19 days
  - Vol_60d: 22 days
  - EMA_5: 15 days
  - BB_Middle: 15 days
  - BB_Upper: 15 days
  - BB_Lower: 16 days
  - Momentum_5d: 11 days
  - Momentum_20d: 22 days

#### B. Runtime Comparison & Observations

* Batch group (rolling windows, mySD + KS-Test):  148.48s
* Stream group (row-by-row, river ADWIN+KSWIN+PH): 10.85s
* River stream group is 13.7x faster

> Despite being row-by-ro